# 🔍 Tahap 1: Preprocessing & Validasi Dataset (Deteksi Sampah Daur Ulang)

**Nama: Fadli Hifizansyah**  
**NIM: 241730042**  
**Kelas: Informatika 4B**  
**Mata Kuliah: Kecerdasan Buatan**  

---

### Deskripsi Tahap Ini:
Pada notebook ini, kita akan melakukan persiapan dan pemeriksaan dataset agar siap digunakan untuk training YOLOv8:
1. Mendeteksi lingkungan kerja secara otomatis (Lokal vs Google Colab).
2. Mengekstrak file dataset jika berjalan di Google Colab.
3. Memvalidasi jumlah file gambar dan label.
4. Membuat file `data.yaml` secara dinamis menggunakan path proyek saat ini.
5. Menganalisis sebaran kelas sampah (kaca, kertas, logam, plastik) dengan grafik.
6. Menampilkan sampel gambar beserta bounding box asli untuk memverifikasi anotasi.

## 🛠️ 1. Setup Lingkungan (Google Colab / Lokal)
Jalankan cell di bawah ini. Jika berjalan di Google Colab, kode akan mengunduh repository dari GitHub dan menginstal dependensi secara otomatis.

In [ ]:
import os
import sys

# Deteksi Google Colab
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print("☁️ Berjalan di Google Colab")
    # Clone repository GitHub jika folder belum ada
    if not os.path.exists('/content/deteksi-sampah-replikasi'):
        print("📥 Cloning repository dari GitHub...")
        !git clone https://github.com/fadli154/deteksi-sampah-replikasi.git
    
    # Pindah ke direktori repository
    %cd /content/deteksi-sampah-replikasi
    
    # Instalasi library yang dibutuhkan
    !pip install pyyaml opencv-python matplotlib numpy
else:
    print("💻 Berjalan di komputer lokal")
    # Pindah ke root folder proyek (naik 2 tingkat dari 05_Source_Code/Notebook)
    %cd ../..
    print(f"Direktori kerja saat ini: {os.getcwd()}")

## 📦 2. Ekstrak Dataset (Khusus Google Colab)
Jika Anda menggunakan Google Colab, silakan **unggah file `dataset.zip`** (yang diunduh dari Roboflow format YOLOv8) ke panel kiri Google Colab, lalu jalankan cell di bawah ini untuk mengekstraknya.

In [ ]:
if IS_COLAB:
    if os.path.exists('/content/dataset.zip'):
        print("📦 Mengekstrak dataset.zip ke folder proyek...")
        !unzip -q /content/dataset.zip -d /content/deteksi-sampah-replikasi/
        print("✅ Dataset berhasil diekstrak!")
    else:
        print("⚠️ dataset.zip tidak ditemukan di folder /content/. ")
        print("Silakan unggah dataset.zip Anda terlebih dahulu melalui panel file di sebelah kiri.")
else:
    print("💻 Anda berada di komputer lokal. Lewati langkah ini (pastikan dataset sudah diekstrak di folder proyek). ")

## 📝 3. Validasi Dataset dan Konfigurasi `data.yaml`
Cell ini akan memeriksa keberadaan folder dataset dan menulis file `data.yaml` dengan path absolut yang disesuaikan secara otomatis.

In [ ]:
import yaml

project_root = os.getcwd()
splits = ['train', 'valid', 'test']
missing_folders = []

# Periksa keberadaan folder train/valid/test
for split in splits:
    if not os.path.exists(os.path.join(project_root, split)):
        missing_folders.append(split)

if missing_folders:
    print(f"❌ Error: Folder berikut tidak ditemukan: {missing_folders}")
    print("Pastikan dataset Anda sudah diekstrak di folder proyek.")
else:
    print("✅ Semua folder dataset (train, valid, test) lengkap!")
    
    # Buat data.yaml secara dinamis sesuai path saat ini
    data_yaml = {
        'path': project_root.replace('\\', '/'),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 4,
        'names': ['kaca', 'kertas', 'logam', 'plastik']
    }
    
    with open('data.yaml', 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
        
    print("✅ data.yaml berhasil dibuat dan dikonfigurasi secara dinamis!")

## 📊 4. Distribusi Kelas Sampah dalam Dataset
Mari kita hitung seberapa banyak objek sampah dari masing-masing kategori (`kaca`, `kertas`, `logam`, `plastik`) di setiap folder pembagian (*split*).

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

class_names = ['kaca', 'kertas', 'logam', 'plastik']
split_counts = {}

for split in splits:
    labels_dir = os.path.join(project_root, split, 'labels')
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))
    
    classes_counter = Counter()
    for lbl_file in label_files:
        try:
            with open(lbl_file, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        classes_counter[class_id] += 1
        except:
            continue
            
    split_counts[split] = [classes_counter.get(i, 0) for i in range(4)]
    print(f"📂 Split [{split.upper()}]:")
    for idx, c_name in enumerate(class_names):
        print(f"   - {c_name.capitalize()}: {split_counts[split][idx]} objek")

# Visualisasi Distribusi Kelas menggunakan Bar Chart
x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, split_counts['train'], width, label='Train', color='#4f46e5')
ax.bar(x, split_counts['valid'], width, label='Validation', color='#0ea5e9')
ax.bar(x + width, split_counts['test'], width, label='Test', color='#10b981')

ax.set_ylabel('Jumlah Objek', fontsize=12)
ax.set_title('Distribusi Kelas Sampah Daur Ulang dalam Dataset', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in class_names], fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 📸 5. Visualisasi Bounding Box Dataset
Jalankan cell ini untuk melihat contoh gambar dari dataset yang digambari kotak pembatas (*bounding box*) anotasi YOLO secara langsung.

In [ ]:
import cv2

def plot_sample_annotations(split='train', num_samples=3):
    images_dir = os.path.join(project_root, split, 'images')
    labels_dir = os.path.join(project_root, split, 'labels')
    
    image_files = glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.jpeg")) + glob.glob(os.path.join(images_dir, "*.png"))
    if not image_files:
        print("Tidak ada gambar ditemukan!")
        return
        
    selected_images = np.random.choice(image_files, min(num_samples, len(image_files)), replace=False)
    
    # Warna untuk masing-masing kelas (B-G-R)
    colors = {
        0: (0, 0, 255),    # kaca -> Merah
        1: (255, 0, 0),    # kertas -> Biru
        2: (0, 255, 255),  # logam -> Kuning
        3: (0, 255, 0)     # plastik -> Hijau
    }
    
    plt.figure(figsize=(15, 6))
    
    for idx, img_path in enumerate(selected_images):
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(labels_dir, f"{base_name}.txt")
        
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        
                        # Hitung koordinat pixel
                        x1 = int((x_c - bw/2) * w)
                        y1 = int((y_c - bh/2) * h)
                        x2 = int((x_c + bw/2) * w)
                        y2 = int((y_c + bh/2) * h)
                        
                        color = colors.get(class_id, (128, 128, 128))
                        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                        cv2.putText(img, class_names[class_id].upper(), (x1, max(y1 - 10, 20)), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                                    
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, num_samples, idx + 1)
        plt.imshow(img_rgb)
        plt.title(f"Sampah: {base_name}", fontsize=11)
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

plot_sample_annotations(split='train', num_samples=3)

## 🏁 Kesimpulan Tahap 1
Dataset Anda berhasil diperiksa dan divalidasi. File `data.yaml` telah disiapkan secara dinamis. Anda siap melanjutkan ke **Tahap 2: Pelatihan Model (`training.ipynb`)**!